In [1]:
!git clone https://github.com/vantuann205/fruit-classification-cnn_yolov5.git

Cloning into 'fruit-classification-cnn_yolov5'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 25 (delta 7), reused 21 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 44.62 MiB | 34.85 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [4]:
%cd /content/fruit-classification-cnn_yolov5

!git fetch origin refs/heads/hieu:refs/remotes/origin/hieu

!git switch --track -c hieu origin/hieu

/content/fruit-classification-cnn_yolov5
branch 'hieu' set up to track 'origin/hieu'.
Switched to a new branch 'hieu'


In [5]:
!git branch -a
!git status
!git branch --show-current

* hieu
  main
  remotes/origin/HEAD -> origin/main
  remotes/origin/hieu
  remotes/origin/main
On branch hieu
Your branch is up to date with 'origin/hieu'.

nothing to commit, working tree clean
hieu


In [ ]:
# !git fetch origin
# !git pull --ff-only origin hieu

In [ ]:
# %cd /content

# !git clone \
#     --branch hieu \
#     --single-branch \
#     https://github.com/vantuann205/fruit-classification-cnn_yolov5.git \
#     fruit-classification-cnn_yolov5-hieu
# %cd /content/fruit-classification-cnn_yolov5-hieu

# !git branch --show-current
# !git log -1 --oneline

# !ls -lah fruit_classfication/

In [11]:
from pathlib import Path
import os, sys, json, math, random, shutil, hashlib, subprocess, zipfile, csv
from collections import Counter, defaultdict
from PIL import Image

In [19]:
import zipfile, shutil
from pathlib import Path

RAW = Path("/content/fruit_raw_extracted")
shutil.rmtree(RAW, ignore_errors=True)

with zipfile.ZipFile("/content/fruit-classification-cnn_yolov5/fruit_classfication/data/Fruit.zip") as zf:
    for m in zf.infolist():
        m.filename = m.filename.replace("\\", "/")
        zf.extract(m, RAW)

CLASS_SOURCES = {label: RAW / "Fruit" / label for label in {"Buoi", "dudu", "Hong"}}

In [20]:
for k, p in CLASS_SOURCES.items():
    print(k, p.is_dir(), len(list(p.rglob("*.jpg"))))

Buoi True 1164
Hong True 1094
dudu True 1602


In [25]:
for label, class_dir in CLASS_SOURCES.items():
    for path in list(class_dir.rglob("*.jpg")):
        if path.parent != class_dir:
            target = class_dir / path.name
            if target.exists():
                target = class_dir / f"{path.stem}_{abs(hash(str(path))) % 10**8}.jpg"
            shutil.move(path, target)

    for sub in sorted((p for p in class_dir.rglob("*") if p.is_dir()),
                      key=lambda p: len(p.parts), reverse=True):
        if not any(sub.iterdir()):
            sub.rmdir()

for label, class_dir in CLASS_SOURCES.items():
    print(label, "->", len(list(class_dir.glob("*.jpg"))), "ảnh ở root |",
          len([p for p in class_dir.iterdir() if p.is_dir()]), "folder con còn lại")

Buoi -> 1164 ảnh ở root | 0 folder con còn lại
Hong -> 1094 ảnh ở root | 0 folder con còn lại
dudu -> 1602 ảnh ở root | 0 folder con còn lại


In [26]:
records, seen = [], {}
for class_name, class_dir in CLASS_SOURCES.items():
    for path in sorted(class_dir.glob("*.jpg")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            continue
        seen[digest] = class_name

        with Image.open(path) as im:
            w, h = im.size
        records.append({"class": class_name, "path": path, "sha256": digest,
                        "width": w, "height": h})

for cn in {"Buoi", "dudu", "Hong"}:
    print(f"  {cn}: {sum(r['class'] == cn for r in records)}")

  Buoi: 1164
  Hong: 1094
  dudu: 1600


In [30]:
PROJECT_DATA_DIR = Path("/content/fruit-classification-cnn_yolov5/fruit_classfication/data")
RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

In [31]:
for sn in RATIO:
    shutil.rmtree(PROJECT_DATA_DIR / sn, ignore_errors=True)
    for cn in {"Buoi", "dudu", "Hong"}:
        (PROJECT_DATA_DIR / sn / cn).mkdir(parents=True, exist_ok=True)


In [33]:
rng = random.Random(SEED)
manifest = []

for cn in {"Buoi", "dudu", "Hong"}:
    items = [r for r in records if r["class"] == cn]
    rng.shuffle(items)
    n = len(items)
    n_tr = int(n * RATIO["train"])
    n_va = int(n * RATIO["val"])
    parts = {"train": items[:n_tr],
             "val":   items[n_tr:n_tr+n_va],
             "test":  items[n_tr+n_va:]}

    for sn, subset in parts.items():
        for rec in subset:
            with Image.open(rec["path"]) as orig:
                im = ImageOps.exif_transpose(orig).convert("RGB")
                w, h = im.size
                s = 256 / max(w, h)
                im = im.resize((max(1, round(w*s)), max(1, round(h*s))), Image.Resampling.LANCZOS)
                out = PROJECT_DATA_DIR / sn / cn / f"{rec['sha256'][:20]}.jpg"
                im.save(out, "JPEG", quality=92, optimize=True)
            manifest.append({"split": sn, "class": cn,
                             "output": str(out.relative_to(PROJECT_DATA_DIR)),
                             "source": str(rec["path"].relative_to(Path("/content/fruit_raw_extracted"))),
                             "source_sha256": rec["sha256"]})

with (PROJECT_DATA_DIR / "split_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=manifest[0].keys())
    w.writeheader(); w.writerows(manifest)

CLASS_TO_IDX = {cn: i for i, cn in enumerate(sorted({"Buoi", "dudu", "Hong"}))}
(PROJECT_DATA_DIR / "class_to_idx.json").write_text(
    json.dumps(CLASS_TO_IDX, ensure_ascii=False, indent=2), encoding="utf-8")

for sn in RATIO:
    print(sn, {cn: len(list((PROJECT_DATA_DIR / sn / cn).glob("*.jpg"))) for cn in CLASS_TO_IDX})
print("CLASS_TO_IDX:", CLASS_TO_IDX)

train {'Buoi': 814, 'Hong': 765, 'dudu': 1120}
val {'Buoi': 174, 'Hong': 164, 'dudu': 240}
test {'Buoi': 176, 'Hong': 165, 'dudu': 240}
CLASS_TO_IDX: {'Buoi': 0, 'Hong': 1, 'dudu': 2}


In [34]:
TRAINVAL_DIR = Path("/content/fruit_trainval")
shutil.rmtree(TRAINVAL_DIR, ignore_errors=True)
shutil.copytree(PROJECT_DATA_DIR / "train", TRAINVAL_DIR / "train")
shutil.copytree(PROJECT_DATA_DIR / "val", TRAINVAL_DIR / "val")
assert not (TRAINVAL_DIR / "test").exists(), "Test leakage!"
print("OK:", TRAINVAL_DIR)

OK: /content/fruit_trainval


In [35]:
YOLOV5_DIR = Path("/content/yolov5")
COMMIT = "33ae15ef99bd04ce1d3f12155b8a85f355d73351"
shutil.rmtree(YOLOV5_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)], check=True)
subprocess.run(["git", "-C", str(YOLOV5_DIR), "checkout", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(YOLOV5_DIR / "requirements.txt")], check=True)
import torch
assert torch.cuda.is_available(), "CUDA mất sau khi cài dependencies — restart runtime"
print("Ready:", COMMIT)

Ready: 33ae15ef99bd04ce1d3f12155b8a85f355d73351


In [36]:
SMOKE_RUN = YOLOV5_DIR / "runs" / "train-cls" / "smoke_1ep"
subprocess.run([
    sys.executable, str(YOLOV5_DIR / "classify" / "train.py"),
    "--model", "yolov5s-cls.pt", "--data", str(TRAINVAL_DIR),
    "--epochs", "1", "--img", "224", "--batch-size", "32",
    "--device", "0", "--workers", "2", "--seed", str(SEED),
    "--project", str(YOLOV5_DIR / "runs" / "train-cls"), "--name", "smoke_1ep",
], cwd=YOLOV5_DIR, check=True)
assert (SMOKE_RUN / "weights" / "best.pt").exists()
print("Smoke OK:", SMOKE_RUN)

Smoke OK: /content/yolov5/runs/train-cls/smoke_1ep


In [40]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/BTL_HocSau/yolov5_cls")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(SMOKE_RUN / "weights" / "best.pt", DRIVE_OUT / "yolov5_cls_smoke_best.pt")
shutil.copy2(SMOKE_RUN / "opt.yaml", DRIVE_OUT / "smoke_opt.yaml")
if (SMOKE_RUN / "results.csv").exists():
    shutil.copy2(SMOKE_RUN / "results.csv", DRIVE_OUT / "smoke_results.csv")
print("Saved:", DRIVE_OUT)

Mounted at /content/drive
Saved: /content/drive/MyDrive/BTL_HocSau/yolov5_cls


### Test trước T08

In [39]:
# RUN_BASELINE = False
RUN_BASELINE = True
if RUN_BASELINE:
    RUN = YOLOV5_DIR / "runs" / "train-cls" / "baseline_30ep"
    subprocess.run([
        sys.executable, str(YOLOV5_DIR / "classify" / "train.py"),
        "--model", "yolov5s-cls.pt", "--data", str(TRAINVAL_DIR),
        "--epochs", "30", "--img", "224", "--batch-size", "32",
        "--device", "0", "--workers", "2", "--seed", str(SEED),
        "--optimizer", "Adam", "--lr0", "0.001",
        "--project", str(YOLOV5_DIR / "runs" / "train-cls"), "--name", "baseline_30ep",
    ], cwd=YOLOV5_DIR, check=True)
    final = Path("/content/fruit-classification-cnn_yolov5/fruit_classfication/models/yolov5_cls_best.pt")
    final.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RUN / "weights" / "best.pt", final)
    print("Baseline saved:", final)

Baseline saved: /content/fruit-classification-cnn_yolov5/fruit_classfication/models/yolov5_cls_best.pt


In [42]:
# RUN_FINAL_TEST = False
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    ckpt = Path("/content/fruit-classification-cnn_yolov5/fruit_classfication/models/yolov5_cls_best.pt")
    assert ckpt.exists(), f"Missing {ckpt}"
    subprocess.run([sys.executable, str(YOLOV5_DIR / "classify" / "val.py"),
                    "--data", str(PROJECT_DATA_DIR), "--weights", str(ckpt),
                    "--img", "224", "--batch-size", "32", "--device", "0", "--verbose"],
                   cwd=YOLOV5_DIR, check=True)
else:
    print("Final test skipped")

In [43]:
print(type(PROJECT_DATA_DIR))

<class 'pathlib._local.PosixPath'>
